# Build a Simple LLM Application with LCEL

In this quickstart, we will learn how to build a simple LLM application with LangChain. This application will translate text from English into another Language. This is a relatively simple LLM application - it's just a single LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call! 

High Level overview of: 
- Using language models
- using promptTemplates and Output Parsers
- Using LangChain Expression Language (LCEL) to chain components together
- Debugging and tracing your application using Langsmith
- Deploying your application with LangServe


In [1]:
# using open source models -- Llama3, Gemma2, mistral --Groq

import os
from dotenv import load_dotenv
load_dotenv()

import openai
openai.api_key = os.getenv("OPENAI_API_KEY")

groq_api_key = os.getenv("GROQ_API_KEY")

In [5]:
from langchain_groq import ChatGroq
model = ChatGroq(model = "llama-3.3-70b-versatile",groq_api_key = groq_api_key)
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x70005efa9ab0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x70005efa9690>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [8]:
from langchain_core.messages import HumanMessage, SystemMessage
messages = [
    SystemMessage(content = "Translate the following from English to German"),
    HumanMessage(content="Hello How are you?")
]

result = model.invoke(messages)

In [9]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()
parser.invoke(result)

'Hallo, wie geht es Ihnen?'

In [11]:
### using LCEL - chain the components
chain = model|parser
chain.invoke(messages)

'Hallo, wie geht es dir?'

In [12]:
### Prompt Templates
from langchain_core.prompts import ChatPromptTemplate
generic_template = "Translate the following into {language}:"
prompt = ChatPromptTemplate.from_messages(
    [("system", generic_template),("user","{text}")]
)


In [15]:
result  = prompt.invoke({"language":"French", "text":"Hello"})

In [16]:
result.to_messages()

[SystemMessage(content='Translate the following into French:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]

In [21]:
#chain with prompt templates
chain = prompt|model|parser
chain.invoke({"language":"German", "text":"Hello"})

'Hallo'